# TabPFN-TS on Real Mechanical System Data

Apply TabPFN-TS to actual mechanical system datasets from this project.

**Datasets covered:**
1. UCI Hydraulic System (industrial process)
2. C-MAPSS (turbofan simulation)
3. Paderborn Bearing (optional, requires download)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pathlib import Path
import sys

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

np.random.seed(42)

## 1. UCI Hydraulic System

Real industrial hydraulic test rig with:
- 17 sensors (pressure, flow, temperature, vibration)
- 2,205 cycles of 60 seconds each
- Multiple fault conditions

In [ ]:
# Check if data exists
hydraulic_path = project_root / 'datasets' / 'data' / 'hydraulic'

if not hydraulic_path.exists():
    print("Hydraulic data not found. Download with:")
    print("  python datasets/downloaders/download_hydraulic.py")
    print("\nUsing synthetic data for demonstration...")
    USE_SYNTHETIC = True
else:
    USE_SYNTHETIC = False
    print(f"Found hydraulic data at: {hydraulic_path}")

In [ ]:
def load_hydraulic_sensor(sensor_name, cycle_idx=0):
    """Load a single sensor for a specific cycle."""
    if USE_SYNTHETIC:
        # Generate synthetic hydraulic-like data
        n = 6000  # 60 seconds at 100 Hz
        t = np.linspace(0, 60, n)
        # Pressure-like signal with transients
        base = 100 + 20 * np.sin(2 * np.pi * 0.05 * t)
        transient = 30 * np.exp(-0.5 * t) * (t < 10)
        noise = 2 * np.random.randn(n)
        return base + transient + noise
    else:
        # Load actual data
        file_path = hydraulic_path / f'{sensor_name}.txt'
        data = np.loadtxt(file_path)
        return data[cycle_idx]

# Load pressure sensor PS1 from first cycle
ps1 = load_hydraulic_sensor('PS1', cycle_idx=0)
print(f"PS1 shape: {ps1.shape}")
print(f"PS1 range: {ps1.min():.2f} - {ps1.max():.2f} bar")

In [ ]:
# Visualize hydraulic pressure signal
t = np.linspace(0, 60, len(ps1))

plt.figure(figsize=(12, 4))
plt.plot(t, ps1, 'b-', linewidth=0.5)
plt.xlabel('Time (s)')
plt.ylabel('Pressure (bar)')
plt.title('Hydraulic System: Pressure Sensor PS1 (Cycle 0)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode

# Subsample for speed (use every 10th point: 10 Hz)
ps1_sub = ps1[::10]
print(f"Subsampled shape: {ps1_sub.shape}")

# Train/test split (first 80% train, last 20% test)
train_size = int(0.8 * len(ps1_sub))
y_train = ps1_sub[:train_size]
y_test = ps1_sub[train_size:]
horizon = len(y_test)

print(f"Train: {len(y_train)}, Test: {len(y_test)}")

In [ ]:
# Prepare data for TabPFN-TS
context_df = pd.DataFrame({
    'item_id': ['PS1'] * len(y_train),
    'timestamp': pd.date_range('2024-01-01', periods=len(y_train), freq='100ms'),
    'target': y_train
})

# Create pipeline and forecast
pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)
pred_df = pipeline.predict_df(context_df=context_df, prediction_length=horizon)
pred_tabpfn = pred_df['target'].values  # Use 'target' column

# Naive baseline (last value)
pred_naive = np.full(horizon, y_train[-1])

# Calculate metrics
rmse_tabpfn = np.sqrt(mean_squared_error(y_test, pred_tabpfn))
rmse_naive = np.sqrt(mean_squared_error(y_test, pred_naive))

print(f"\nHydraulic PS1 Forecasting Results:")
print(f"  TabPFN-TS RMSE: {rmse_tabpfn:.4f}")
print(f"  Naive RMSE:     {rmse_naive:.4f}")
print(f"  Improvement:    {100*(rmse_naive - rmse_tabpfn)/rmse_naive:.1f}%")

In [ ]:
# Visualize
t_sub = np.linspace(0, 60, len(ps1_sub))

plt.figure(figsize=(12, 5))
plt.plot(t_sub[:train_size], y_train, 'b-', label='Training', linewidth=0.8)
plt.plot(t_sub[train_size:], y_test, 'g-', label='True', linewidth=1.5)
plt.plot(t_sub[train_size:], pred_tabpfn, 'r--', label=f'TabPFN-TS (RMSE={rmse_tabpfn:.2f})', linewidth=1.5)
plt.plot(t_sub[train_size:], pred_naive, 'k:', label=f'Naive (RMSE={rmse_naive:.2f})', linewidth=1.5)
plt.axvline(x=t_sub[train_size], color='gray', linestyle=':', alpha=0.7)
plt.xlabel('Time (s)')
plt.ylabel('Pressure (bar)')
plt.title('Hydraulic Pressure Forecasting: TabPFN-TS vs Naive')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. C-MAPSS Turbofan Engine

Simulated turbofan degradation with:
- 21 sensors + 3 operating settings
- Run-to-failure trajectories
- Multiple units and fault modes

In [ ]:
# Check for C-MAPSS data
cmapss_path = project_root / 'datasets' / 'data' / 'cmapss'

if not cmapss_path.exists():
    print("C-MAPSS data not found. Download with:")
    print("  python datasets/downloaders/download_cmapss.py")
    print("\nUsing synthetic data for demonstration...")
    USE_SYNTHETIC_CMAPSS = True
else:
    USE_SYNTHETIC_CMAPSS = False
    print(f"Found C-MAPSS data at: {cmapss_path}")

In [ ]:
def load_cmapss_unit(unit_id=1, fd_subset='FD001'):
    """Load sensor data for a single engine unit."""
    if USE_SYNTHETIC_CMAPSS:
        # Synthetic degradation trajectory
        n_cycles = 150
        t = np.arange(n_cycles)
        
        # Temperature sensor with degradation trend
        base_temp = 600 + 50 * np.random.randn()
        degradation = 0.1 * t + 0.001 * t**2
        noise = 5 * np.random.randn(n_cycles)
        sensor = base_temp + degradation + noise
        
        # Operating settings as covariates
        op_setting_1 = np.random.choice([0, 10, 20, 25, 35, 42], n_cycles)
        op_setting_2 = np.random.choice([0.84, 0.89, 1.00], n_cycles)
        
        return sensor, np.column_stack([op_setting_1, op_setting_2])
    else:
        # Load actual data
        train_file = cmapss_path / f'train_{fd_subset}.txt'
        data = pd.read_csv(train_file, sep=' ', header=None)
        # Columns: unit_id, cycle, op1, op2, op3, s1, s2, ..., s21
        unit_data = data[data[0] == unit_id]
        sensor = unit_data[7].values  # s2 = T24 (LPC outlet temperature)
        settings = unit_data[[2, 3, 4]].values
        return sensor, settings

# Load unit 1 from FD001
temp_sensor, op_settings = load_cmapss_unit(unit_id=1)
print(f"Sensor shape: {temp_sensor.shape}")
print(f"Settings shape: {op_settings.shape}")

In [ ]:
# Visualize turbofan data
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(temp_sensor, 'b-', linewidth=0.8)
axes[0].set_ylabel('Temperature (°R)')
axes[0].set_title('C-MAPSS: LPC Outlet Temperature (Unit 1)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(op_settings[:, 0], 'r-', label='Setting 1', linewidth=0.8)
axes[1].plot(op_settings[:, 1] * 50, 'g-', label='Setting 2 (scaled)', linewidth=0.8)
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Operating Settings')
axes[1].set_title('Operating Conditions (Potential Covariates)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Forecast with and without covariates
train_size = int(0.7 * len(temp_sensor))
y_train = temp_sensor[:train_size]
y_test = temp_sensor[train_size:]
X_train = op_settings[:train_size]
X_test = op_settings[train_size:]
horizon = len(y_test)

print(f"Train: {train_size} cycles, Test: {horizon} cycles")

In [ ]:
# Helper function for forecasting with optional covariates
def forecast_cmapss(y_train, y_test, X_train=None, X_test=None):
    """Forecast C-MAPSS data with optional covariates."""
    context = pd.DataFrame({
        'item_id': ['unit1'] * len(y_train),
        'timestamp': pd.date_range('2024-01-01', periods=len(y_train), freq='D'),
        'target': y_train
    })
    
    pipe = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)
    
    if X_train is not None and X_test is not None:
        # Add covariates
        for i in range(X_train.shape[1]):
            context[f'setting_{i}'] = X_train[:, i]
        
        # Future with covariates
        last_ts = context['timestamp'].iloc[-1]
        future = pd.DataFrame({
            'item_id': ['unit1'] * len(X_test),
            'timestamp': pd.date_range(last_ts + pd.Timedelta(days=1), periods=len(X_test), freq='D'),
        })
        for i in range(X_test.shape[1]):
            future[f'setting_{i}'] = X_test[:, i]
        
        pred_df = pipe.predict_df(context_df=context, future_df=future)
    else:
        pred_df = pipe.predict_df(context_df=context, prediction_length=len(y_test))
    
    return pred_df['target'].values  # Use 'target' column

# Without covariates
pred_no_cov = forecast_cmapss(y_train, y_test)

# With operating settings as covariates
pred_with_cov = forecast_cmapss(y_train, y_test, X_train, X_test)

# Naive baselines
pred_naive = np.full(horizon, y_train[-1])
pred_trend = y_train[-1] + np.arange(1, horizon + 1) * np.mean(np.diff(y_train[-10:]))

# Results
results = {
    'TabPFN-TS (no cov)': np.sqrt(mean_squared_error(y_test, pred_no_cov)),
    'TabPFN-TS (with cov)': np.sqrt(mean_squared_error(y_test, pred_with_cov)),
    'Naive (last value)': np.sqrt(mean_squared_error(y_test, pred_naive)),
    'Naive (linear trend)': np.sqrt(mean_squared_error(y_test, pred_trend)),
}

print("\nC-MAPSS Temperature Forecasting Results:")
print("-" * 40)
for name, rmse in results.items():
    print(f"{name:25s}: RMSE = {rmse:.4f}")

In [ ]:
# Visualize
cycles = np.arange(len(temp_sensor))

plt.figure(figsize=(12, 5))
plt.plot(cycles[:train_size], y_train, 'b-', label='Training', linewidth=0.8)
plt.plot(cycles[train_size:], y_test, 'g-', label='True', linewidth=1.5)
plt.plot(cycles[train_size:], pred_no_cov, 'r--', label='TabPFN (no cov)', linewidth=1.5)
plt.plot(cycles[train_size:], pred_with_cov, 'm--', label='TabPFN (with cov)', linewidth=1.5)
plt.plot(cycles[train_size:], pred_trend, 'k:', label='Linear trend', linewidth=1.5)
plt.axvline(x=train_size, color='gray', linestyle=':', alpha=0.7)
plt.xlabel('Cycle')
plt.ylabel('Temperature (°R)')
plt.title('C-MAPSS Temperature Forecasting with Degradation Trend')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Summary Table

Compile results across datasets for comparison.

In [ ]:
summary_data = {
    'Dataset': ['Hydraulic PS1', 'C-MAPSS T24'],
    'TabPFN-TS': [rmse_tabpfn, results['TabPFN-TS (no cov)']],
    'TabPFN-TS (cov)': ['-', results['TabPFN-TS (with cov)']],
    'Naive': [rmse_naive, results['Naive (last value)']],
    'Best': ['TabPFN-TS' if rmse_tabpfn < rmse_naive else 'Naive',
             'TabPFN-TS (cov)' if results['TabPFN-TS (with cov)'] < results['Naive (last value)'] else 'Naive'],
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("Summary: TabPFN-TS on Mechanical Systems")
print("="*60)
print(summary_df.to_string(index=False))

## 4. Key Observations

Based on these experiments:

1. **Periodic signals (Hydraulic)**: TabPFN-TS should capture periodic patterns reasonably well

2. **Degradation trends (C-MAPSS)**: The model needs to extrapolate trends, which is challenging for any method

3. **Covariates**: Operating conditions as covariates can help when they genuinely affect the target

4. **Comparison to baselines**: The key question is whether TabPFN-TS beats simple baselines—this determines practical usefulness

## 5. Next Steps

See `experiments/` folder for:
- Systematic comparison across all datasets
- More sophisticated baselines (ARIMA, Prophet)
- Cross-condition transfer experiments

## Exercises

### Exercise 1: Multiple Sensors

Load multiple sensors from the hydraulic system and compare TabPFN-TS performance across different sensor types (pressure vs temperature vs flow).

In [ ]:
# TODO: Load PS1, TS1, FS1 and compare forecasting performance
# ...

### Exercise 2: Cross-Sensor Covariates

Use one sensor as a covariate to predict another (e.g., use pressure to predict flow).

In [ ]:
# TODO: Load PS1 and FS1, use PS1 as covariate to predict FS1
# ...

### Exercise 3: Different Forecast Horizons

How does TabPFN-TS performance degrade with longer horizons on mechanical data?

In [ ]:
# TODO: Test horizons [10, 50, 100, 200] and plot RMSE vs horizon
# ...